In [0]:
%sql
WITH ultima_versao AS (SELECT nct_id,
                              payload,
                              ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY collected_at DESC, page_number DESC) AS rn
                         FROM mvp_eng_dados.mvp_cancer.brz_clinical_trials),

         registros AS (SELECT nct_id,
                              TRY_CAST(get_json_object(payload,'$.protocolSection.statusModule.studyFirstPostDateStruct.date') AS DATE) AS data_primeira_publicacao
                         FROM ultima_versao
                        WHERE rn = 1)
                        
SELECT YEAR(data_primeira_publicacao) AS ano_registro,
       COUNT(DISTINCT nct_id) AS quantidade_estudos
  FROM registros
 GROUP BY YEAR(data_primeira_publicacao)
 ORDER BY ano_registro NULLS LAST;

In [0]:
%sql
WITH estudos_por_pais AS (SELECT c.country_iso3,
                                 c.country_name,
                                 COUNT(DISTINCT b.study_key) AS quantidade_estudos
                            FROM mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_location b
                            JOIN mvp_eng_dados.mvp_cancer.gld_dim_country c
                                ON b.country_key = c.country_key
                            GROUP BY c.country_iso3, c.country_name)

SELECT DENSE_RANK() OVER (ORDER BY quantidade_estudos DESC) AS posicao,
       country_iso3 AS codigo_iso3,
       country_name AS pais,
       quantidade_estudos
  FROM estudos_por_pais
 ORDER BY posicao, pais;

In [0]:
%sql
WITH base AS (SELECT nct_id,
                     COALESCE(NULLIF(TRIM(phase), ''), 'NA') AS fase
                FROM mvp_eng_dados.mvp_cancer.slv_studies)

SELECT fase,
       COUNT(DISTINCT nct_id) AS quantidade_estudos,
       ROUND(100.0 * COUNT(DISTINCT nct_id) / SUM(COUNT(DISTINCT nct_id)) OVER (), 2) AS percentual
  FROM base
 GROUP BY fase
 ORDER BY quantidade_estudos DESC;

In [0]:
%sql
SELECT intervention_type AS tipo_intervencao,
       COUNT(DISTINCT nct_id) AS quantidade_estudos
   FROM mvp_eng_dados.mvp_cancer.slv_interventions
  GROUP BY intervention_type
  ORDER BY quantidade_estudos DESC;

In [0]:
%sql
SELECT d.intervention_type AS tipo_intervencao,
       d.intervention_name AS intervencao,
       COUNT(DISTINCT b.study_key) AS quantidade_estudos
FROM mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_intervention b
JOIN mvp_eng_dados.mvp_cancer.gld_dim_intervention d
    ON b.intervention_key = d.intervention_key
GROUP BY
    d.intervention_type,
    d.intervention_name
ORDER BY quantidade_estudos DESC, intervencao
LIMIT 20;

In [0]:
%sql
SELECT overall_status AS situacao,
       COUNT(DISTINCT nct_id) AS quantidade_estudos,
       ROUND(100.0 * COUNT(DISTINCT nct_id) / SUM(COUNT(DISTINCT nct_id)) OVER (), 2) AS percentual
  FROM mvp_eng_dados.mvp_cancer.slv_studies
 GROUP BY overall_status
 ORDER BY quantidade_estudos DESC;

In [0]:
%sql
SELECT
    COALESCE(NULLIF(TRIM(phase), ''), 'NA') AS fase,
    study_type AS tipo_estudo,
    COALESCE(enrollment_type, 'NA') AS tipo_contagem,
    COUNT(*) AS quantidade_estudos,
    COUNT(enrollment_count) AS estudos_com_participantes,
    ROUND(AVG(enrollment_count), 2) AS media_participantes,
    percentile_approx(enrollment_count, 0.5) AS mediana_participantes
FROM mvp_eng_dados.mvp_cancer.slv_studies
WHERE COALESCE(enrollment_type, 'NA') <> 'NA'
GROUP BY COALESCE(NULLIF(TRIM(phase), ''), 'NA'),
         study_type,
         COALESCE(enrollment_type, 'NA')
ORDER BY fase, tipo_estudo, tipo_contagem;

In [0]:
%sql
SELECT d.sponsor_name AS patrocinador,
       d.sponsor_class AS categoria_patrocinador,
       COUNT(DISTINCT f.study_key) AS quantidade_estudos
  FROM mvp_eng_dados.mvp_cancer.gld_fat_clinical_study f
  JOIN mvp_eng_dados.mvp_cancer.gld_dim_sponsor d
    ON f.sponsor_key = d.sponsor_key
 GROUP BY d.sponsor_key,
          d.sponsor_name,
          d.sponsor_class
 ORDER BY quantidade_estudos DESC, 
          patrocinador
LIMIT 20;

In [0]:
%sql
WITH estudos_elegiveis AS ( SELECT COALESCE(NULLIF(TRIM(phase), ''), 'SEM_INFORMACAO') AS fase,
                                   enrollment_count,
                                   duration_days
                              FROM mvp_eng_dados.mvp_cancer.slv_studies
                             WHERE study_type = 'INTERVENTIONAL'
                               AND overall_status = 'COMPLETED'
                               AND enrollment_type = 'ACTUAL'
                               AND start_date_type = 'ACTUAL'
                               AND completion_date_type = 'ACTUAL'
                               AND LENGTH(start_date_original) = 10
                               AND LENGTH(completion_date_original) = 10
                               AND enrollment_count IS NOT NULL
                               AND duration_days IS NOT NULL)

SELECT fase,
       COUNT(*) AS estudos_elegiveis,
       ROUND(AVG(enrollment_count), 2) AS media_participantes,
       percentile_approx(enrollment_count, 0.5) AS mediana_participantes,
       ROUND(AVG(duration_days), 2) AS media_duracao_dias,
       percentile_approx(duration_days, 0.5) AS mediana_duracao_dias,
       ROUND(CORR(enrollment_count, duration_days),4) AS correlacao_participantes_duracao
  FROM estudos_elegiveis
 GROUP BY fase
HAVING COUNT(*) >= 3
 ORDER BY fase;

In [0]:
%sql
WITH parametros AS (SELECT 2025 AS ano_referencia),

base AS (SELECT m.year,
                m.country_iso3,
                m.country_name,
                m.study_count,
                m.population,
                CASE WHEN m.population > 0 THEN 1000000.0 * m.study_count / m.population END AS estudos_por_milhao
           FROM mvp_eng_dados.mvp_cancer.gld_flat_country_year_metrics m
     CROSS JOIN parametros p
          WHERE m.year = p.ano_referencia)

SELECT year AS ano_inicio,
    country_iso3 AS codigo_iso3,
    country_name AS pais,
    CASE WHEN country_iso3 = 'BRA' THEN 'BRASIL' ELSE 'OUTROS PAISES' END AS identificacao,
    study_count AS quantidade_estudos,
    population AS populacao,
    ROUND(estudos_por_milhao, 4) AS estudos_por_milhao,
    DENSE_RANK() OVER (ORDER BY study_count DESC) AS ranking_quantidade,
    CASE WHEN estudos_por_milhao IS NOT NULL THEN DENSE_RANK() OVER (ORDER BY estudos_por_milhao DESC NULLS LAST) END AS ranking_por_milhao
FROM base
ORDER BY ranking_quantidade, pais;

In [0]:
%sql
WITH estudos_por_pais_ano AS (SELECT b.country_key,
                                     dt.year AS ano,
                                     COUNT(DISTINCT f.study_key) AS quantidade_estudos,
                                     COUNT(DISTINCT CASE WHEN s.phase IN ('PHASE3', 'PHASE2|PHASE3')THEN f.study_key END) AS estudos_fase3
                                FROM mvp_eng_dados.mvp_cancer.gld_fat_clinical_study f
                                JOIN mvp_eng_dados.mvp_cancer.gld_dim_study s
                                  ON f.study_key = s.study_key
                                JOIN mvp_eng_dados.mvp_cancer.gld_dim_date dt
                                  ON f.start_date_key = dt.date_key
                                JOIN mvp_eng_dados.mvp_cancer.gld_flat_bridge_study_location b
                                  ON f.study_key = b.study_key
                                WHERE dt.year IN (2024, 2025)
                                  AND s.study_type = 'INTERVENTIONAL'
                                GROUP BY b.country_key, dt.year),

    indicadores AS (SELECT e.country_key,
                           e.ano,
                           e.quantidade_estudos,
                           e.estudos_fase3,
                           p.population AS populacao,
                           1000000.0 * e.quantidade_estudos
                               / p.population AS estudos_por_milhao,
                           100.0 * e.estudos_fase3
                               / e.quantidade_estudos AS percentual_fase3
                      FROM estudos_por_pais_ano e
                      JOIN mvp_eng_dados.mvp_cancer.gld_fat_country_population p
                        ON e.country_key = p.country_key
                       AND p.date_key = e.ano * 10000 + 101
                     WHERE p.population > 0
                       AND e.quantidade_estudos >= 10),

    comparacao AS (SELECT atual.country_key,
                          anterior.quantidade_estudos AS estudos_2024,
                          atual.quantidade_estudos AS estudos_2025,
                          anterior.estudos_por_milhao AS taxa_2024,
                          atual.estudos_por_milhao AS taxa_2025,
                          100.0 * (atual.estudos_por_milhao / NULLIF(anterior.estudos_por_milhao, 0) - 1) AS crescimento_percentual,
                          anterior.percentual_fase3 AS fase3_percentual_2024,
                          atual.percentual_fase3 AS fase3_percentual_2025,
                          atual.percentual_fase3- anterior.percentual_fase3 AS variacao_fase3_pp
                     FROM indicadores atual
                     JOIN indicadores anterior
                       ON atual.country_key = anterior.country_key
                      AND anterior.ano = atual.ano - 1
                    WHERE atual.ano = 2025)

SELECT DENSE_RANK() OVER ( ORDER BY c.crescimento_percentual DESC) AS ranking_crescimento,
       pais.country_iso3 AS codigo_iso3,
       pais.country_name AS pais,
       c.estudos_2024,
       c.estudos_2025,
       ROUND(c.taxa_2024, 4) AS estudos_por_milhao_2024,
       ROUND(c.taxa_2025, 4) AS estudos_por_milhao_2025,
       ROUND(c.crescimento_percentual, 2) AS crescimento_percentual,
       ROUND(c.fase3_percentual_2024, 2) AS percentual_fase3_2024,
       ROUND(c.fase3_percentual_2025, 2) AS percentual_fase3_2025,
       ROUND(c.variacao_fase3_pp, 2) AS variacao_fase3_pontos_percentuais
  FROM comparacao c
  JOIN mvp_eng_dados.mvp_cancer.gld_dim_country pais
    ON c.country_key = pais.country_key
 ORDER BY ranking_crescimento, 
          pais;